# Урок 11 - Протокол агент-към-агент (A2A)


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Какво представлява протоколът A2A?

Протоколът **Agent-to-Agent (A2A)** е отворен стандарт, който позволява на AI агенти да комуникират,
да се откриват взаимно и да си сътрудничат — дори когато са изградени върху различни рамки или хоствани
от различни услуги.

Key concepts:

- **Откриване** – Агентите публикуват *Картичка на агента*, която описва техните възможности, улеснявайки
  другите агенти (или оркестратори) да намерят правилния специалист за задача.
- **Обмен на съобщения** – Агентите разменят структурирани съобщения чрез общ протокол, така че
  заявка от един агент да може да бъде разбрана и изпълнена от друг, независимо от неговата вътрешна
  реализация.
- **Животен цикъл на задачата** – A2A дефинира състояния като *подадено*, *в процес*, *завършено* и
  *неуспешно*, давайки на оркестратора пълна видимост за това как напредва възложената задача.

В този урок симулираме сътрудничество в A2A стил, като свързваме три специализирани туристически агента
в работен поток, в който всеки агент допринася със своята експертиза и предава резултатите на следващия.


## Създаване на специализирани туристически агенти


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Многоагентно сътрудничество чрез работен поток

Свързваме трите агента в последователен работен поток, който отразява A2A предаването на съобщения:

1. **CurrencyExchangeAgent** получава потребителското запитване и предоставя указания относно валутите.
2. **ActivityPlannerAgent** получава обогатения контекст и добавя препоръки за дейности.
3. **TravelManagerAgent** синтезира и двата входа в окончателен план за пътуване.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Разбиране на A2A в производствена среда

В производствена среда протоколът A2A отключва мощни сценарии между услуги:

| Възможност | Описание |
|---|---|
| **Съвместимост между рамки** | Агент, изграден с една рамка, може да делегира задачи на агент, изграден с която и да е друга рамка, съвместима с A2A, позволявайки истинска съвместимост между организации. |
| **Граници на услугите** | Агентите могат да се намират в отделни микроуслуги, облачни региони или дори различни организации, като все пак си сътрудничат безпроблемно. |
| **Динамично откриване** | Оркестратор може да направи заявка към регистъра на Agent Card по време на изпълнение, за да намери най-подходящия специалист за дадена подзадача. |
| **Поточно предаване и push известия** | A2A поддържа Server-Sent Events (SSE) за актуализации на напредъка в реално време и push известия за дълго изпълняващи се задачи. |

Работният поток, който изградихме по-горе, е опростена, изпълнявана в рамките на един процес версия на този модел. В реално
разгръщане всеки агент би изложил HTTP крайна точка, би публикувал Agent Card и би комуникирал
чрез протокола A2A JSON-RPC.


## Резюме

В този урок научихте:

1. **Какво представлява A2A протоколът** — отворен стандарт за агент-към-агент откриване, съобщения,
   и управление на задачи.
2. **Как да създадете специализирани агенти** — агент за обмен на валута, агент за планиране на дейности,
   и оркестратор за управление на пътувания.
3. **Как да свържете агентите в работен поток** — използвайки `WorkflowBuilder` за моделиране на последователно
   предаване на съобщения между агентите.
4. **Как A2A работи в продукция** — позволява сътрудничество между различни рамки и различни услуги
   с динамично откриване и потокови актуализации.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI услуга за превод [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля, имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ в оригиналния му език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален превод, извършен от квалифициран преводач. Не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
